# **Part 1 - Setup & Imports**

In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch
from pydantic import BaseModel
from typing import Literal

Bro API KEY Variable exists


## **Part 2 - Schema + Base LLM**

In [2]:
class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=300)

## **Part 3 - Main Prompt + Structured Output + Extractor**


In [3]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Categorize the movie review as positive or negative: {input}")
])

llm_structured_output = llm.with_structured_output(llm_schema)

def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

## **Part 4 - LinkedIn Chain (positive)**

In [4]:
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator. Output ONLY the post. Be concise, max 150 words."),
    ("human", "{text}")
])

chain_linkedin = linkedin_prompt | ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=300) | StrOutputParser()

## **Part 5 - Instagram Chain (negative)**

In [5]:
def insta_chain(text: str) -> str:
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator. Output ONLY the post. Be concise, max 100 words."),
        ("human", "{text}")
    ])
    chain_insta = insta_prompt | ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=200) | StrOutputParser()
    return chain_insta.invoke({"text": text})

insta_chain_runnable = RunnableLambda(insta_chain)

## **Part 6 - Final Orchestration + Run**

In [ ]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

topic = input("Enter a movie review: ")
result = final_orchestrator.invoke({"input": topic})
print(result)

"Spread positivity wherever you go. Believe in yourself, take risks, and never give up on your dreams. Surround yourself with people who uplift and support you. Focus on the good, and watch your life transform. Remember, a positive mindset is the key to unlocking your full potential. Share with me in the comments below what keeps you motivated and inspired to tackle each day with a positive attitude! #PositiveVibes #Motivation #Inspiration"


: 